# CloudTrend 미국주식 전용 스크리닝 수집기 · Prospective V1

한국 `cloudtrend자료수집_v3.2_스크리닝.ipynb`의 **Colab Secrets → Drive cache → Supabase publish** 구조를 참고하되, 미국주식은 별도 데이터 계약으로 수집합니다.

### 고정 운용/검증 구조
- **실제운용(Primary): A0 분기** — Core `ret120/ret252` 50:50, E80 Onset, X70 또는 beta rank < 0.60 연속 3거래일 Anchor, beta top10%, TK gap top20%, sector cap 없음, 최대 20종목, 분기 첫 거래일 비중조정.
- **Shadow 1: A2 분기** — 기존 E80/X70 신호 (A0 Anchor 미적용) + 동일 섹터 최대 2종목.
- **Shadow 2: B3 Beta** — E80, X50 + beta rank < 0.60 3일, relvol top20%, 동일 섹터 최대 3종목.
- 이 노트북은 **횡단면 순위·Onset·Exit를 계산하지 않습니다.** 원자 피처만 Supabase에 올리고 GitHub 엔진이 한 번만 계산합니다.

### Toss Open API 사용
- OAuth2 Client Credentials: `/oauth2/token`
- 종목 마스터: `/api/v1/stocks`
- Adjusted 일봉: `/api/v1/candles?interval=1d`
- 현재가: `/api/v1/prices` (진단용)
- US 장 캘린더: `/api/v1/market-calendar/US`
- USD/KRW: `/api/v1/exchange-rate`
- US 랭킹: `/api/v1/rankings` (진단용, 전략 점수에는 사용하지 않음)

국내 전용 투자자매매/프로그램/공매도/신용/대차 API는 미국 스크리닝 입력에서 제외합니다.


In [ ]:
# ===== 1. 사용자 설정 =====
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/CloudTrend/미국시장/us_prospective')
CACHE_PATH = BASE_DIR / 'us_toss_daily.parquet'
MASTER_PATH = BASE_DIR / 'us_toss_master.parquet'
OUTPUT_DIR = BASE_DIR / 'outputs'

# 전략 계산을 위해 최소 252일 + warm-up이 필요합니다.
INITIAL_BARS = 400
INCREMENTAL_BARS = 15
MAX_WORKERS = 8
CHART_RPS = 12.0  # 공식 한도보다 보수적으로 시작. 실제 응답 rate-limit 헤더를 우선 확인하세요.

# 현재 prospective universe seed는 US3.8 mapping_v2의 current common-equity eligible 목록입니다.
# strict PIT를 만들지 않으며, 앞으로의 일별 스크리닝 대상만 고정/갱신합니다.
FORCE_FULL_REFRESH = False
DISPATCH_GITHUB_AFTER_UPLOAD = True
GITHUB_REPO = 'Jiwoon625/cloudtrend'
GITHUB_WORKFLOW = 'us-prospective-screening.yml'

BASE_DIR, CACHE_PATH


In [ ]:
# ===== 2. 설치 · Drive 마운트 · Colab Secrets =====
!pip -q install pandas==2.2.3 pyarrow==19.0.1 requests==2.32.5 tqdm==4.67.1

import os, json, time, math, hashlib, threading, glob
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from google.colab import drive, userdata

drive.mount('/content/drive')
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOSS_CLIENT_ID = userdata.get('TOSS_CLIENT_ID')
TOSS_CLIENT_SECRET = userdata.get('TOSS_CLIENT_SECRET')
SUPABASE_URL = 'https://ahbvrtugugwnbrfnbxzp.supabase.co'
SUPABASE_SERVICE_ROLE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
SUPABASE_USER_ID = userdata.get('SUPABASE_USER_ID')
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None

required = {
    'TOSS_CLIENT_ID': TOSS_CLIENT_ID,
    'TOSS_CLIENT_SECRET': TOSS_CLIENT_SECRET,
    'SUPABASE_SERVICE_ROLE_KEY': SUPABASE_SERVICE_ROLE_KEY,
    'SUPABASE_USER_ID': SUPABASE_USER_ID,
}
missing = [k for k,v in required.items() if not v]
if missing:
    raise RuntimeError(f'Colab Secrets 누락: {missing}')

print('Secrets OK:', ', '.join(required))


In [2]:
# ===== 3. 현재 Colab 공인 IP 확인 (Toss 허용 IP 등록용) =====

import requests

def get_public_ip():
    endpoints = [
        "https://api.ipify.org",
        "https://checkip.amazonaws.com",
    ]

    errors = []

    for endpoint in endpoints:
        try:
            response = requests.get(endpoint, timeout=10)
            response.raise_for_status()

            ip = response.text.strip()

            if ip:
                return ip
        except Exception as error:
            errors.append(f"{endpoint}: {error}")

    raise RuntimeError(
        "외부 IP 확인 실패\n" + "\n".join(errors)
    )


COLAB_PUBLIC_IP = get_public_ip()

print("현재 실행환경의 외부 IP:", COLAB_PUBLIC_IP)
print("이 IP가 Toss Open API 허용 IP에 등록되어 있어야 합니다.")

현재 실행환경의 외부 IP: 35.192.4.139
이 IP가 Toss Open API 허용 IP에 등록되어 있어야 합니다.


In [ ]:
# ===== 4. Toss OAuth2 · 재시도 · rate limiter =====
BASE_URL = 'https://openapi.tossinvest.com'

def issue_token():
    r = requests.post(
        f'{BASE_URL}/oauth2/token',
        headers={'Content-Type':'application/x-www-form-urlencoded'},
        data={
            'grant_type':'client_credentials',
            'client_id':TOSS_CLIENT_ID,
            'client_secret':TOSS_CLIENT_SECRET,
        }, timeout=30,
    )
    r.raise_for_status()
    body = r.json()
    token = body.get('access_token') or body.get('accessToken') or body.get('result',{}).get('accessToken')
    if not token:
        raise RuntimeError('OAuth response has no access_token')
    return token

_token_lock = threading.Lock()
TOKEN = issue_token()
SESSION = requests.Session()
SESSION.headers.update({'Authorization': f'Bearer {TOKEN}', 'Accept':'application/json'})

_rate_lock = threading.Lock()
_last_chart = 0.0

def _chart_throttle():
    global _last_chart
    with _rate_lock:
        wait = max(0.0, (1.0/CHART_RPS) - (time.monotonic()-_last_chart))
        if wait: time.sleep(wait)
        _last_chart = time.monotonic()

def toss_get(path, params=None, chart=False, retries=6):
    global TOKEN
    for attempt in range(retries):
        if chart: _chart_throttle()
        used_token = TOKEN
        try:
            r = SESSION.get(BASE_URL+path, params=params or {}, timeout=45)
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(min(16, 2**attempt)); continue
        if r.status_code == 401 and attempt == 0:
            with _token_lock:
                if TOKEN == used_token:
                    TOKEN = issue_token()
                    SESSION.headers['Authorization'] = f'Bearer {TOKEN}'
            continue
        if r.status_code == 429:
            retry = float(r.headers.get('Retry-After') or min(8, 1.5**attempt))
            time.sleep(retry); continue
        if r.status_code in (500,502,503,504):
            time.sleep(min(8, 1.5**attempt)); continue
        r.raise_for_status()
        return r.json(), dict(r.headers)
    raise RuntimeError(f'Toss GET 재시도 소진: {path} {params}')

print('OAuth token issued')


In [ ]:
# ===== 5. Prospective US universe seed 구성 =====
# 연구 당시 mapping_v2의 common-equity 정의를 유지하되, 매일 Toss /stocks/all 로
# 현재 NYSE/NASDAQ/AMEX ACTIVE 종목인지 교차검증합니다. strict PIT 재구성은 하지 않습니다.

REFERENCE_MAPPING = Path('/content/drive/MyDrive/CloudTrend/미국시장/reference/mapping_v2_20260927_r2.csv')
if REFERENCE_MAPPING.exists():
    mapping_path = str(REFERENCE_MAPPING)
else:
    raise FileNotFoundError(str(REFERENCE_MAPPING))
print('mapping:', mapping_path)
mp = pd.read_csv(mapping_path, dtype=str, keep_default_na=False)

def truthy(s): return str(s).strip().lower() in {'true','1','y','yes','t'}
mask = mp['commonEquityEligible'].map(truthy)
if 'isdelisted' in mp.columns:
    mask &= mp['isdelisted'].str.upper().ne('Y')
if 'securityType' in mp.columns:
    mask &= mp['securityType'].str.upper().eq('COMMON_EQUITY')
mapping_seed = mp.loc[mask, ['ticker','sectorCode']].drop_duplicates('ticker').copy()
mapping_seed['ticker'] = mapping_seed['ticker'].str.upper().str.strip()
mapping_seed = mapping_seed[mapping_seed['ticker'].str.match(r'^[A-Z0-9.\-]+$')]

# Toss가 현재 거래가능하다고 제공하는 미국 상장 종목과 교차검증.
listed=[]
for market in ['NYSE','NASDAQ','AMEX']:
    body,_ = toss_get('/api/v1/stocks/all', {'market':market,'status':'ACTIVE','commonShare':'true'})
    result = body.get('result', body.get('data', body))
    if isinstance(result, dict): result = result.get('stocks', result.get('items', []))
    if isinstance(result, list):
        for item in result:
            item=dict(item); item['market']=market; listed.append(item)
    time.sleep(1.05)  # STOCK_ALL 그룹은 보수적으로 1초 간격
listed_df=pd.DataFrame(listed)
if listed_df.empty or 'symbol' not in listed_df.columns:
    raise RuntimeError('Toss /api/v1/stocks/all 미국 유니버스 조회가 비었습니다.')
listed_df['symbol']=listed_df['symbol'].astype(str).str.upper().str.strip()
toss_active=set(listed_df['symbol'])
seed=mapping_seed[mapping_seed['ticker'].isin(toss_active)].copy()

# SPY는 전략 종목이 아니라 beta/benchmark용으로 항상 포함
if 'SPY' not in set(seed.ticker):
    seed = pd.concat([seed, pd.DataFrame([{'ticker':'SPY','sectorCode':'BENCHMARK'}])], ignore_index=True)
seed_path = BASE_DIR / 'us_universe_seed.csv'
seed.to_csv(seed_path, index=False)
print('mapping eligible:',len(mapping_seed),'Toss active listed:',len(toss_active),'prospective intersection:',len(seed))
print('seed saved:',seed_path)


In [ ]:
# ===== 6. Toss 종목 마스터 검증/보강 =====
def chunks(values, n=200):
    values=list(values)
    for i in range(0,len(values),n): yield values[i:i+n]

master_rows=[]
for batch in tqdm(list(chunks(seed.ticker.tolist(), 200)), desc='stock master'):
    body,_ = toss_get('/api/v1/stocks', {'symbols': ','.join(batch)})
    result = body.get('result', body.get('data', body))
    if isinstance(result, dict): result = result.get('stocks', result.get('items', []))
    if not isinstance(result, list): result=[]
    master_rows.extend(result)
    time.sleep(0.22)  # STOCK group conservative pacing

master = pd.DataFrame(master_rows)
if master.empty:
    raise RuntimeError('Toss /api/v1/stocks 결과가 비었습니다. 허용 IP/토큰/응답 스키마를 확인하세요.')
# response field aliases tolerant
for col in ['symbol','name','englishName','market','securityType','status','currency','listDate','delistDate','sharesOutstanding','isCommonShare']:
    if col not in master.columns: master[col] = np.nan
master['symbol'] = master['symbol'].astype(str).str.upper()
master = master.drop_duplicates('symbol', keep='last')
master = master.merge(seed.rename(columns={'ticker':'symbol','sectorCode':'sector'}), on='symbol', how='right')
master['status'] = master['status'].fillna('UNKNOWN')
master['isCommonShare'] = master['isCommonShare'].fillna(False)
if master.status.eq('UNKNOWN').any():
    raise RuntimeError('Incomplete master; retry before publishing')
master.to_parquet(MASTER_PATH, index=False)
print('master rows:', len(master), 'ACTIVE:', int(master.status.astype(str).str.upper().eq('ACTIVE').sum()))


In [ ]:
# ===== 7. Adjusted 일봉 수집/증분 캐시 =====
from zoneinfo import ZoneInfo
now_utc = pd.Timestamp.now(tz='UTC')
calendar_body, _ = toss_get('/api/v1/market-calendar/US', {'date': now_utc.tz_convert('America/New_York').date().isoformat()})
calendar_days = list(calendar_body['result'].values())
closed = [d for d in calendar_days if d.get('regularMarket') and pd.Timestamp(d['regularMarket']['endTime']) + pd.Timedelta(minutes=30) <= now_utc]
if not closed: raise RuntimeError('No confirmed US regular-session close')
AS_OF_DATE = max(d['date'] for d in closed)
asof_calendar, _ = toss_get('/api/v1/market-calendar/US', {'date': AS_OF_DATE})
PREVIOUS_SESSION = asof_calendar['result']['previousBusinessDay']['date']
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / AS_OF_DATE
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Confirmed session:', AS_OF_DATE)

def parse_candles(symbol, body):
    result = body.get('result', body.get('data', body))
    if isinstance(result, dict): candles = result.get('candles', result.get('items', []))
    elif isinstance(result, list): candles = result
    else: candles=[]
    out=[]
    for x in candles:
        ts = x.get('timestamp') or x.get('date')
        if not ts: continue
        out.append({
            'symbol':symbol,
            'date':str(ts)[:10],
            'open':pd.to_numeric(x.get('openPrice',x.get('open')),errors='coerce'),
            'high':pd.to_numeric(x.get('highPrice',x.get('high')),errors='coerce'),
            'low':pd.to_numeric(x.get('lowPrice',x.get('low')),errors='coerce'),
            'close':pd.to_numeric(x.get('closePrice',x.get('close')),errors='coerce'),
            'volume':pd.to_numeric(x.get('volume'),errors='coerce'),
            'currency':x.get('currency'),
        })
    next_before = result.get('nextBefore') if isinstance(result,dict) else None
    return out,next_before

def fetch_history(symbol, full=False):
    need = INITIAL_BARS if full else INCREMENTAL_BARS
    rows=[]; before=None; seen=set()
    while len(rows)<need:
        params={'symbol':symbol,'interval':'1d','count':200,'adjusted':'true'}
        if before: params['before']=before
        body,_=toss_get('/api/v1/candles',params,chart=True)
        part,next_before=parse_candles(symbol,body)
        if not part: break
        unique = [r for r in part if r['date'] not in seen and r['date'] <= AS_OF_DATE]
        rows.extend(unique)
        seen.update(r['date'] for r in unique)
        if not next_before or next_before==before: break
        before=next_before
    return rows[:INITIAL_BARS] if len(rows)>INITIAL_BARS else rows

if CACHE_PATH.exists() and not FORCE_FULL_REFRESH:
    old = pd.read_parquet(CACHE_PATH)
    old['symbol']=old['symbol'].astype(str).str.upper()
    counts=old.groupby('symbol').size().to_dict()
else:
    old = pd.DataFrame(columns=['symbol','date','open','high','low','close','volume','currency'])
    counts={}

symbols=seed.ticker.tolist()
collected=[]; failures=[]

def task(sym):
    checkpoint = CHECKPOINT_DIR / (sym + '.json')
    try:
        if checkpoint.exists() and not FORCE_FULL_REFRESH:
            return sym, json.loads(checkpoint.read_text()), None
        # Re-fetch the full feature window: adjusted history may change after corporate actions.
        rows = fetch_history(sym, full=True)
        if not rows: raise RuntimeError('empty candle history')
        tmp = checkpoint.with_suffix('.tmp')
        tmp.write_text(json.dumps(rows, allow_nan=False))
        tmp.replace(checkpoint)
        return sym, rows, None
    except Exception as e: return sym, [], type(e).__name__ + ': ' + str(e)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs=[ex.submit(task,s) for s in symbols]
    for f in tqdm(as_completed(futs), total=len(futs), desc='daily candles'):
        sym, rows, err=f.result()
        if err: failures.append((sym,err))
        else: collected.extend(rows)

new=pd.DataFrame(collected)
# Each successful symbol replaces its adjusted feature window, avoiding split discontinuities.
combined=pd.concat([old[~old.symbol.isin(new.symbol.unique())] if not new.empty else old,new],ignore_index=True)
combined=combined.dropna(subset=['symbol','date','close']).drop_duplicates(['symbol','date'],keep='last')
combined=combined.sort_values(['symbol','date']).reset_index(drop=True)
combined.to_parquet(CACHE_PATH,index=False,compression='zstd')
print('cache rows:',len(combined),'symbols:',combined.symbol.nunique(),'failures:',len(failures))
if failures: print('failure examples:',failures[:10])

(BASE_DIR/'collection_failures.json').write_text(json.dumps(failures))
if failures: raise RuntimeError(f'{len(failures)} symbols failed; checkpoints saved. Re-run before publishing.')


In [ ]:
# ===== 8. 전략 원자 피처 계산 =====
px=combined[combined.date.le(AS_OF_DATE) & combined.symbol.isin(seed.ticker)].copy()
for c in ['open','high','low','close','volume']:
    px[c]=pd.to_numeric(px[c],errors='coerce')
px=px.sort_values(['symbol','date'])
g=px.groupby('symbol',group_keys=False)
px['ret1']=g['close'].pct_change()
px['ret120']=g['close'].pct_change(120)
px['ret252']=g['close'].pct_change(252)
px['dollar_volume']=px['close']*px['volume']
px['adv20_usd']=g['dollar_volume'].transform(lambda s:s.rolling(20,min_periods=20).mean())
px['amihud_raw']=px['ret1'].abs()/px['dollar_volume'].replace(0,np.nan)
px['amihud20']=g['amihud_raw'].transform(lambda s:s.rolling(20,min_periods=20).mean())
px['vol20_avg']=g['volume'].transform(lambda s:s.rolling(20,min_periods=20).mean())
px['relvol1_20']=px['volume']/px['vol20_avg']-1
px['tenkan9']=(g['high'].transform(lambda s:s.rolling(9,min_periods=9).max())+g['low'].transform(lambda s:s.rolling(9,min_periods=9).min()))/2
px['kijun26']=(g['high'].transform(lambda s:s.rolling(26,min_periods=26).max())+g['low'].transform(lambda s:s.rolling(26,min_periods=26).min()))/2
px['ichimoku_tk_gap']=px['tenkan9']/px['kijun26']-1
px['active20']=g['volume'].transform(lambda s:s.gt(0).rolling(20,min_periods=20).sum()).ge(20)

# beta60_spy: 종목 일수익률과 SPY 일수익률의 최근 60개 공통 관측 cov/var
spy=px[px.symbol.eq('SPY')][['date','ret1']].rename(columns={'ret1':'spy_ret1'})
px=px.merge(spy,on='date',how='left')
def beta60(frame):
    cov=frame['ret1'].rolling(60,min_periods=50).cov(frame['spy_ret1'])
    var=frame['spy_ret1'].rolling(60,min_periods=50).var()
    frame['beta60_spy']=cov/var.replace(0,np.nan)
    return frame
px=pd.concat([beta60(frame.copy()) for _,frame in px.groupby('symbol')],ignore_index=True)

latest_date=AS_OF_DATE
if px.loc[px.symbol.eq('SPY'),'date'].max() != latest_date: raise RuntimeError('SPY is stale')
snap=px[px.date.eq(latest_date)].copy()
# 마스터 보강
mcols=['symbol','name','englishName','market','securityType','status','currency','sharesOutstanding','isCommonShare','sector']
meta=master[[c for c in mcols if c in master.columns]].copy()
snap=snap.merge(meta,on='symbol',how='left',suffixes=('','_master'))
if 'currency_master' in snap: snap['currency']=snap['currency_master'].fillna(snap['currency'])
snap['shares_outstanding']=pd.to_numeric(snap.get('sharesOutstanding'),errors='coerce')
snap['market_cap']=snap['close']*snap['shares_outstanding']
snap['toss_tradable']=snap['status'].astype(str).str.upper().eq('ACTIVE')
snap['is_common_share']=snap['isCommonShare'].fillna(False).map(truthy) & snap.symbol.ne('SPY')
# Missing current bars must be retried, never converted into mass universe exits.
missing_current=set(seed.ticker)-set(snap.symbol)
if missing_current: raise RuntimeError(f'Missing confirmed-session bars: {sorted(missing_current)[:20]}')
snap['name']=snap.get('englishName').fillna(snap.get('name')).fillna(snap['symbol'])
print('latest date:',latest_date,'snapshot:',len(snap),'complete ret252:',snap.ret252.notna().sum())


In [ ]:
# ===== 9. US 캘린더 · 환율 · 현재가 · 랭킹 snapshot (진단용) =====
def safe_get(path, params=None):
    try:
        body,headers=toss_get(path,params)
        return {'ok':True,'body':body,'rateLimit':{k:v for k,v in headers.items() if 'rate' in k.lower() or 'limit' in k.lower()}}
    except Exception as e:
        return {'ok':False,'error':str(e)}

market_calendar={'ok':True,'body':asof_calendar}
exchange_rate=safe_get('/api/v1/exchange-rate',{'baseCurrency':'USD','quoteCurrency':'KRW'})
# 공식 Ranking API는 전략에는 사용하지 않고 운영 진단 snapshot만 보존합니다.
ranking_amount=safe_get('/api/v1/rankings',{'marketCountry':'US','type':'MARKET_TRADING_AMOUNT','duration':'1d','count':100})
ranking_gainers=safe_get('/api/v1/rankings',{'marketCountry':'US','type':'TOP_GAINERS','duration':'1d','count':100})

# 현재가는 종가 확정 여부/운영 QA용. 200개씩 다건 지원.
price_rows=[]
for batch in chunks(seed.ticker.tolist(),200):
    try:
        body,_=toss_get('/api/v1/prices',{'symbols':','.join(batch)})
        result=body.get('result',[])
        if isinstance(result,list): price_rows.extend(result)
    except Exception:
        pass
    time.sleep(0.08)

# fx 숫자는 응답 형태가 바뀔 수 있어 원문을 metadata에 남기고, 찾을 수 있을 때만 row에 기록
fx_usdkrw=np.nan
def find_number(obj):
    if isinstance(obj,dict):
        for k,v in obj.items():
            if str(k).lower() in {'rate','exchangerate','usdkrw','price'}:
                try: return float(v)
                except: pass
        for v in obj.values():
            x=find_number(v)
            if x is not None:return x
    if isinstance(obj,list):
        for v in obj:
            x=find_number(v)
            if x is not None:return x
    return None
if exchange_rate.get('ok'):
    fx_usdkrw=find_number(exchange_rate['body']) or np.nan
snap['fx_usdkrw']=fx_usdkrw

market_meta={
    'collectedAt':datetime.now(timezone.utc).isoformat(),
    'asOfDate':latest_date,
    'marketCalendar':market_calendar,
    'exchangeRate':exchange_rate,
    'rankingMarketTradingAmount':ranking_amount,
    'rankingTopGainers':ranking_gainers,
    'currentPriceSampleCount':len(price_rows),
}
(BASE_DIR/'market_meta_latest.json').write_text(json.dumps(market_meta,ensure_ascii=False,indent=2,default=str))
print('market meta saved')


In [ ]:
# ===== 10. Supabase용 atomic screening snapshot 생성 =====
cols={
    'date':'date','symbol':'symbol','name':'name','market':'market','sector':'sector','securityType':'security_type','status':'status','currency':'currency',
    'open':'open','high':'high','low':'low','close':'close','volume':'volume','dollar_volume':'dollar_volume','shares_outstanding':'shares_outstanding','market_cap':'market_cap',
    'ret120':'ret120','ret252':'ret252','beta60_spy':'beta60_spy','ichimoku_tk_gap':'ichimoku_tk_gap','relvol1_20':'relvol1_20','adv20_usd':'adv20_usd','amihud20':'amihud20',
    'active20':'active20','toss_tradable':'toss_tradable','is_common_share':'is_common_share','fx_usdkrw':'fx_usdkrw'
}
out=snap[[c for c in cols if c in snap.columns]].rename(columns=cols).copy()
# SPY는 benchmark용으로 포함. 엔진은 SPY를 종목 후보에서는 제외합니다.
out=out.sort_values('symbol').reset_index(drop=True)
out_path=OUTPUT_DIR/f'us_screening_input_{latest_date.replace("-","")}.csv'
out=out.replace([np.inf,-np.inf],np.nan)
assert out.symbol.is_unique and out.date.eq(AS_OF_DATE).all()
assert out.loc[out.symbol.eq('SPY'),'close'].gt(0).all()
out.to_csv(out_path,index=False,lineterminator='\n')
raw=out_path.read_bytes(); data_hash='sha256:'+hashlib.sha256(raw).hexdigest()
print('saved:',out_path,'rows:',len(out),'sha:',data_hash)
display(out.head())


In [ ]:
# ===== 11. Supabase private Storage + ingest pointer 업로드 =====
assert SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY and SUPABASE_USER_ID
storage_path=f'{SUPABASE_USER_ID}/source/us-screening/{latest_date}/{data_hash.split(":")[1]}.csv'
headers={
    'Authorization':f'Bearer {SUPABASE_SERVICE_ROLE_KEY}',
    'apikey':SUPABASE_SERVICE_ROLE_KEY,
}
up=requests.post(
    f'{SUPABASE_URL}/storage/v1/object/cloudtrend-data/{storage_path}',
    headers={**headers,'Content-Type':'text/csv','x-upsert':'false'},
    data=raw, timeout=120,
)
if up.status_code>=300 and up.status_code not in (400,409):
    raise RuntimeError(f'Supabase Storage upload failed {up.status_code}: {up.text[:1000]}')

readback=requests.get(f'{SUPABASE_URL}/storage/v1/object/authenticated/cloudtrend-data/{storage_path}',headers=headers,timeout=120)
readback.raise_for_status()
assert hashlib.sha256(readback.content).hexdigest() == data_hash.split(':')[1], 'Storage hash mismatch'
import io
verified=pd.read_csv(io.BytesIO(readback.content))
assert len(verified)==len(out) and verified.symbol.nunique()==len(out) and verified.date.eq(latest_date).all()
prior=requests.get(f'{SUPABASE_URL}/rest/v1/us_screening_ingest',headers=headers,params={'user_id':'eq.'+SUPABASE_USER_ID,'select':'as_of_date,data_hash'},timeout=30)
prior.raise_for_status()
if prior.json():
    existing=prior.json()[0]
    if existing['as_of_date']>latest_date or (existing['as_of_date']==latest_date and existing['data_hash']!=data_hash):
        raise RuntimeError('Existing date/hash is immutable; source saved separately for QA')

payload={
    'user_id':SUPABASE_USER_ID,
    'as_of_date':latest_date,
    'storage_bucket':'cloudtrend-data',
    'storage_path':storage_path,
    'row_count':int(len(out)),
    'symbol_count':int(out.symbol.nunique()),
    'data_hash':data_hash,
    'schema_version':'us-prospective-v1',
    'source_provider':'TOSS_OPEN_API',
    'collected_at':datetime.now(timezone.utc).isoformat(),
    'metadata':{
        'notebook':'cloudtrend_US_스크리닝자료수집_v1.ipynb',
        'cachePath':str(CACHE_PATH),
        'masterPath':str(MASTER_PATH),
        'mappingVersion':'20260927_r2',
        'failedSymbols':len(failures),
        'previousSessionDate':PREVIOUS_SESSION,
        'confirmedRegularClose':True,
        'collectorVersion':'1.1.0',
        'marketCalendarOk':bool(market_calendar.get('ok')),
        'exchangeRateOk':bool(exchange_rate.get('ok')),
        'rankingOk':bool(ranking_amount.get('ok')),
    }
}
r=requests.post(
    f'{SUPABASE_URL}/rest/v1/us_screening_ingest?on_conflict=user_id',
    headers={**headers,'Content-Type':'application/json','Prefer':'resolution=merge-duplicates,return=representation'},
    json=payload, timeout=60,
)
if r.status_code>=300: raise RuntimeError(f'ingest upsert failed {r.status_code}: {r.text}')
print('Supabase upload OK:',storage_path)

check=requests.get(f'{SUPABASE_URL}/rest/v1/us_screening_ingest',headers=headers,params={'user_id':'eq.'+SUPABASE_USER_ID},timeout=30)
check.raise_for_status()
assert check.json()[0]['data_hash']==data_hash
print('Storage + ingest read-back verified')


In [ ]:
# ===== 12. GitHub Actions 엔진 실행 트리거 =====
if DISPATCH_GITHUB_AFTER_UPLOAD and GITHUB_TOKEN:
    url=f'https://api.github.com/repos/{GITHUB_REPO}/actions/workflows/{GITHUB_WORKFLOW}/dispatches'
    rr=requests.post(url,headers={'Authorization':f'Bearer {GITHUB_TOKEN}','Accept':'application/vnd.github+json'},json={'ref':'main','inputs':{'supabase_user_id':SUPABASE_USER_ID}},timeout=30)
    if rr.status_code not in (200,201,204):
        raise RuntimeError(f'GitHub workflow dispatch 실패 {rr.status_code}: {rr.text[:1000]}')
    print('GitHub US prospective workflow dispatched')
else:
    print('즉시 실행 token 없음: GitHub의 정기 US workflow가 업로드를 자동 처리합니다.')


In [ ]:
# ===== 13. 실행 요약 =====
print('=== CloudTrend US prospective collection complete ===')
print('as-of:',latest_date)
print('universe seed:',len(seed))
print('snapshot rows:',len(out))
print('history cache rows:',len(combined))
print('collection failures:',len(failures))
print('Supabase:',storage_path)
print('data hash:',data_hash)
print('다음 단계: GitHub 엔진이 A0 Primary / A2+B3 Shadow 신호·포트폴리오를 계산해 Supabase와 웹에 반영합니다.')
